**Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones**

**Exploración y Curación de Datos**

*Edición 2023*

----

# Trabajo práctico entregable - parte 1

## **RESOLUCIÓN EJERCICIO 1.1: Crear db**

1. Crear una base de datos en SQLite utilizando la libreria [SQLalchemy](https://stackoverflow.com/questions/2268050/execute-sql-from-file-in-sqlalchemy).
https://docs.sqlalchemy.org/en/14/core/engines.html#sqlite

In [ ]:
# download Northwind SQLite DB (this is needed in google colab)
!wget https://tdmdal.github.io/mma-sql-2021/data/northwind.sqlite3

--2023-05-30 22:30:59--  https://tdmdal.github.io/mma-sql-2021/data/northwind.sqlite3
Resolving tdmdal.github.io (tdmdal.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to tdmdal.github.io (tdmdal.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 804864 (786K) [application/octet-stream]
Saving to: ‘northwind.sqlite3’

northwind.sqlite3   100%[===================>] 786.00K  --.-KB/s    in 0.009s  

2023-05-30 22:30:59 (81.3 MB/s) - ‘northwind.sqlite3’ saved [804864/804864]



In [ ]:
! pip install ipython-sql==0.5.0
! pip install sqlalchemy==2.0.15

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.6 MB/s eta 0:00:00
  Attempting uninstall: ipython-sql
    Found existing installation: ipython-sql 0.4.1
    Uninstalling ipython-sql-0.4.1:
      Successfully uninstalled ipython-sql-0.4.1
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 42.2 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.10
    Uninstalling SQLAlchemy-2.0.10:
      Successfully uninstalled SQLAlchemy-2.0.10


In [1]:
import pandas as pd
from sqlalchemy import create_engine
from matplotlib.ticker import FuncFormatter

/home/ruben/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
def human_format(num, pos=1):
    """
    human formatting for big numbers
    """
    magnitude = 0
    while abs(num) >= 1000:
        magnitude += 1
        num /= 1000.0
    return "%.0f%s" % (num, ["", "K", "M", "G", "T", "P"][magnitude])


FORMATTER = FuncFormatter(human_format)

In [ ]:
# (del) the engine object connects a Pool and Dialect together to provide a source of database connectivity and behavior.

*(del)*  
![](https://docs.sqlalchemy.org/en/20/_images/sqla_engine_arch.png)

> Si bien no es lo ideal escribir/leer datos desde una db vía jupyter (sino contar con la instancia correspondiente y credenciales de autenticación, pudiendo configurar accesos y otras configuraciones de seguridad), por simplicidad, facilidad de lectura y limpieza del código se emplea el *magic de sql* para poder interactuar con su consola desde la celda.

In [3]:
%load_ext sql

In [4]:
! export DATABASE_URL=sqlite:///melbourne.sqlite3

In [5]:
%%sql 
sqlite:///melbourne.sqlite3

In [6]:
%%sql
SELECT sqlite_version();

 * sqlite:///melbourne.sqlite3
Done.


sqlite_version()
3.31.1


In [7]:
engine = create_engine('sqlite:///melbourne.sqlite3', echo=True)
engine

Engine(sqlite:///melbourne.sqlite3)

## **RESOLUCIÓN EJERCICIO 1.2: df to sql**

2. Ingestar los datos provistos en 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv' en una tabla y el dataset generado en clase con datos de airbnb y sus precios por codigo postal en otra.

### Ingestión de datos

#### Ingesta de datos de Melbourne

In [8]:
melbourne_data_url = 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv'

df = pd.read_csv(melbourne_data_url)
df.head(10).style.format(
    {'Price': human_format,},
    precision=1,
    na_rep='missing'
    ).highlight_null(color='gray'
    ).bar(subset=['Price'], color='green'
    ).bar(subset=['BuildingArea'], color='brown')

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1M,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,missing,missing,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1M,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1M,SP,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850K,PI,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,1.0,94.0,missing,missing,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,2M,VB,Nelson,4/06/2016,2.5,3067.0,3.0,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
5,Abbotsford,129 Charles St,2,h,941K,S,Jellis,7/05/2016,2.5,3067.0,2.0,1.0,0.0,181.0,missing,missing,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
6,Abbotsford,124 Yarra St,3,h,2M,S,Nelson,7/05/2016,2.5,3067.0,4.0,2.0,0.0,245.0,210.0,1910.0,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
7,Abbotsford,98 Charles St,2,h,2M,S,Nelson,8/10/2016,2.5,3067.0,2.0,1.0,2.0,256.0,107.0,1890.0,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
8,Abbotsford,6/241 Nicholson St,1,u,300K,S,Biggin,8/10/2016,2.5,3067.0,1.0,1.0,1.0,0.0,missing,missing,Yarra,-37.8,145.0,Northern Metropolitan,4019.0
9,Abbotsford,10 Valiant St,2,h,1M,S,Biggin,8/10/2016,2.5,3067.0,3.0,1.0,2.0,220.0,75.0,1900.0,Yarra,-37.8,145.0,Northern Metropolitan,4019.0


In [9]:
df

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.79960,144.99840,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.80790,144.99340,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.80930,144.99440,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.79690,144.99690,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.80720,144.99410,Northern Metropolitan,4019.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,Wheelers Hill,12 Strada Cr,4,h,1245000.0,S,Barry,26/08/2017,16.7,3150.0,...,2.0,2.0,652.0,NaN,1981.0,NaN,-37.90562,145.16761,South-Eastern Metropolitan,7392.0
13576,Williamstown,77 Merrett Dr,3,h,1031000.0,SP,Williams,26/08/2017,6.8,3016.0,...,2.0,2.0,333.0,133.0,1995.0,NaN,-37.85927,144.87904,Western Metropolitan,6380.0
13577,Williamstown,83 Power St,3,h,1170000.0,S,Raine,26/08/2017,6.8,3016.0,...,2.0,4.0,436.0,NaN,1997.0,NaN,-37.85274,144.88738,Western Metropolitan,6380.0
13578,Williamstown,96 Verdon St,4,h,2500000.0,PI,Sweeney,26/08/2017,6.8,3016.0,...,1.0,5.0,866.0,157.0,1920.0,NaN,-37.85908,144.89299,Western Metropolitan,6380.0


In [10]:
df.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'],
      dtype='object')

In [11]:
# (del) df.to_sql('<table_name>', engine, ...)
df.to_sql('melbourne_housing', engine, if_exists='replace'); #(nombre de la tabla, 'engine', 'if_exists=replace: si existe esta tabla, reemplazala)

2023-06-04 19:22:07,252 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2023-06-04 19:22:07,263 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("melbourne_housing")
2023-06-04 19:22:07,273 INFO sqlalchemy.engine.Engine [raw sql] ()
2023-06-04 19:22:07,301 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("melbourne_housing")
2023-06-04 19:22:07,316 INFO sqlalchemy.engine.Engine [raw sql] ()
2023-06-04 19:22:07,319 INFO sqlalchemy.engine.Engine ROLLBACK
2023-06-04 19:22:07,330 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2023-06-04 19:22:07,342 INFO sqlalchemy.engine.Engine 
CREATE TABLE melbourne_housing (
	"index" BIGINT, 
	"Suburb" TEXT, 
	"Address" TEXT, 
	"Rooms" BIGINT, 
	"Type" TEXT, 
	"Price" FLOAT, 
	"Method" TEXT, 
	"SellerG" TEXT, 
	"Date" TEXT, 
	"Distance" FLOAT, 
	"Postcode" FLOAT, 
	"Bedroom2" FLOAT, 
	"Bathroom" FLOAT, 
	"Car" FLOAT, 
	"Landsize" FLOAT, 
	"BuildingArea" FLOAT, 
	"YearBuilt" FLOAT, 
	"CouncilArea" TEXT, 
	"Lattitude" FLOAT, 
	"Longtitude" FLOAT, 

| Column | Description | Type |
|---|---|---|
| Suburb | Self explanitory | str |
| Address | Self explanitory | str |
| Rooms | Number of rooms | int |
| Price | Price in Australian dollars | float |
| Method | S - property sold;   SP - property sold prior;   PI - property passed in;   PN - sold prior not disclosed;   SN - sold not disclosed;   NB - no bid;   VB - vendor bid;   W - withdrawn prior to auction;   SA - sold after auction;   SS - sold after auction price not disclosed.   N/A - price or highest bid not available.   | str (should be cat) |
| Type | br - bedroom(s);   h - house,cottage,villa, semi,terrace;   u - unit, duplex;   t - townhouse;   dev site - development site;   o res - other residential.   | str (should be cat) |
| SellerG | Real Estate Agent | str |
| Date | Date sold | str (should be datetime) |
| Distance | Distance from CBD in Kilometres | float |
| Regionname | General Region (West, North West, North, North east …etc) | str (should be cat) |
| Propertycount | Number of properties that exist in the suburb. | int |
| Bedroom2 | Scraped # of Bedrooms (from different source) | int |
| Bathroom | Number of Bathrooms | int |
| Car | Number of carspots | int |
| Landsize | Land Size in Metres | float |
| BuildingArea | Building Size in Metres | float |
| YearBuilt | Year the house was built | int |
| CouncilArea | Governing council for the area | str (should be cat) |
| Lattitude | Self explanitory | float |
| Longtitude | Self explanitory | float |

#### Ingesta de datos de Airbnb 


In [12]:
df_by_zip_code = pd.read_csv('data/airbnb_price_by_zipcode.csv')
df_by_zip_code.head(10).style.format(
    {col: human_format for col in ['airbnb_price_mean', 'airbnb_weekly_price_mean', 'airbnb_monthly_price_mean']},
    precision=1,
    na_rep='missing'
    ).highlight_null(color='gray'
    ).bar(subset=['airbnb_price_mean', 'airbnb_weekly_price_mean', 'airbnb_monthly_price_mean'], color='green')

,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
0,2010.0,40,1,missing,missing
1,2134.0,50,1,missing,missing
2,2582.0,104,1,missing,missing
3,3000.0,151,3367,919,3K
4,3001.0,132,2,missing,missing
5,3002.0,201,197,956,4K
6,3003.0,130,267,760,3K
7,3004.0,158,728,1K,4K
8,3006.0,189,1268,1K,4K
9,3008.0,177,616,1K,3K


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
df_by_zip_code.to_sql('melbourne_housing_by_zipcode', engine, if_exists='replace')

2023-06-04 19:23:00,964 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2023-06-04 19:23:00,979 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("melbourne_housing_by_zipcode")
2023-06-04 19:23:00,984 INFO sqlalchemy.engine.Engine [raw sql] ()
2023-06-04 19:23:00,990 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("melbourne_housing_by_zipcode")
2023-06-04 19:23:01,003 INFO sqlalchemy.engine.Engine [raw sql] ()
2023-06-04 19:23:01,008 INFO sqlalchemy.engine.Engine ROLLBACK
2023-06-04 19:23:01,020 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2023-06-04 19:23:01,033 INFO sqlalchemy.engine.Engine 
CREATE TABLE melbourne_housing_by_zipcode (
	"index" BIGINT, 
	zipcode FLOAT, 
	airbnb_price_mean FLOAT, 
	airbnb_record_count BIGINT, 
	airbnb_weekly_price_mean FLOAT, 
	airbnb_monthly_price_mean FLOAT
)


2023-06-04 19:23:01,038 INFO sqlalchemy.engine.Engine [no key 0.00490s] ()
2023-06-04 19:23:01,078 INFO sqlalchemy.engine.Engine CREATE INDEX ix_melbourne_housing_by_zipcode_index O

247

## **RESOLUCIÓN EJERCICIO 1.3: Consulta sql**

Implementar consultas en SQL que respondan con la siguiente información:

    - cantidad de registros totales por ciudad.
    - cantidad de registros totales por barrio y ciudad.

El [dataset provisto](https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv) no contiene información sobre la ciudad correspondiente a cada propiedad, 
los docentes de la materia establecen la siguiente equivalencia:  
![](https://i.imgur.com/Y0LmwZS.png)  
https://diplodatos2023espacio.slack.com/archives/C050H0P4ZLK/p1685377293142099

Sin embargo, se considera que este mapeo es incorrecto debido a las siguientes causas:

- SUBURBIO "es un término propio de la geografía urbana, (...), para designar a las zonas residenciales de la periferia urbana o extrarradio", fuente: https://es.wikipedia.org/wiki/Suburbio 
   - Algunas de las entradas de la columna suburbio son: 'Abbotsford', 'Airport West', 'Albert Park', 'Alphington', 'Altona', 'Altona North', 'Armadale', 'Ascot Vale', 'Ashburton',
       - Abbotsford (primer ocurrencia de la columna `Suburb`: "Formerly part of the City of Collingwood, it is now part of the City of Yarra", fuente: https://en.wikipedia.org/wiki/Abbotsford,_Victoria
- A diferencia de lo mencionado anteriormente, las entradas de la columna CIUDAD --> `city` ('Manningham', 'Moreland', 'Port Phillip', 'Darebin', 'Casey') **SI** corresponden a ciudades, y las mismas se encuentran en el dataset https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/cleansed_listings_dec18.csv (también provisto por los docentes de la materia).
   - Por ejemplo: "The City of Manningham is a local government area in Victoria", fuente: https://en.wikipedia.org/wiki/City_of_Manningham

 Es decir: suburb ≠ ciudad.

- Respecto a Regionname, un ejemplo de entrada es <i>North Metropolitan Region</i>: "The North Metropolitan Region is a multi-member electoral region of the Western Australian Legislative Council", fuente: https://en.wikipedia.org/wiki/North_Metropolitan_Region_(Western_Australia)

En otras palabras, se debe tener en cuenta que la distribución regional de Australia no es semejante a la de Argentina y que no se tienen conocimientos de dominio en el tema, por lo cual, cualquier mapeo hecho arbitrariamente sería erróneo. Como máximo, se podría considerar a 'suburbio' como un equivalente de 'barrio'.

Por cuestiones de optimización de tiempo y simplicidad, se utiliza la equivalencia propuesta por la docente, aunque se considere incorrecta.

(del) 

**Cláusula SELECT**:

<i>En las consultas SQL, la cláusula SELECT se utiliza para especificar las columnas que se desean incluir en el resultado de la consulta. </i> 

En este caso, se seleccionaron las columnas de interes, como "CouncilArea" para la consulta de cantidad de registros totales por ciudad y "Suburb" y "CouncilArea" para la consulta de cantidad de registros totales por barrio y ciudad.
También se utilizó la función de agregación COUNT(*) para contar el número de registros en cada grupo.

__Cláusula FROM__:

<i>La cláusula FROM se utiliza para especificar la tabla o tablas de las cuales se obtendrán los datos. </i>

En este caso, se especificó la tabla "melbourne_housing" como la fuente de datos.

**Cláusula GROUP BY**:

<i>La cláusula GROUP BY se utiliza para agrupar los resultados de la consulta según una o más columnas.</i>

En la primera consulta, se agruparon los registros por la columna "CouncilArea" para obtener la cantidad de registros totales por ciudad.
En la segunda consulta, se agruparon los registros por las columnas "Suburb" y "CouncilArea" para obtener la cantidad de registros totales por barrio y ciudad.

**Función de agregación COUNT(*)**:

<i>La función de agregación COUNT(*) se utiliza para contar el número de registros en cada grupo definido por la cláusula GROUP BY.</i>

En ambos casos, se utilizó COUNT(*) para contar el número total de registros en cada grupo.



En resumen, las consultas utilizan la cláusula GROUP BY para agrupar los registros por ciudad y por barrio y ciudad, respectivamente, y luego utilizan la función de agregación COUNT(*) para contar la cantidad de registros en cada grupo. Esto permite obtener la cantidad de registros totales por ciudad y por barrio y ciudad en el dataset 'melbourne_housing'.

In [14]:
%sql SELECT * FROM melbourne_housing LIMIT 2;

 * sqlite:///melbourne.sqlite3
Done.


index,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,None,None,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0


In [15]:
%%sql entries_by_suburb << 
SELECT Suburb, COUNT(*) AS TotalRecords
FROM melbourne_housing
GROUP BY Suburb;

 * sqlite:///melbourne.sqlite3
Done.
Returning data to local variable entries_by_suburb


In [16]:
entries_by_suburb

Suburb,TotalRecords
Abbotsford,56
Aberfeldie,44
Airport West,67
Albanvale,6
Albert Park,69
Albion,41
Alphington,34
Altona,74
Altona Meadows,6
Altona North,56


In [17]:
%%sql entries_by_suburb_and_region << 
SELECT Suburb, Regionname, COUNT(*) AS TotalRecords
FROM melbourne_housing
GROUP BY Suburb, Regionname;

 * sqlite:///melbourne.sqlite3
Done.
Returning data to local variable entries_by_suburb_and_region


In [18]:
entries_by_suburb_and_region

Suburb,Regionname,TotalRecords
Abbotsford,Northern Metropolitan,56
Aberfeldie,Western Metropolitan,44
Airport West,Western Metropolitan,67
Albanvale,Western Metropolitan,6
Albert Park,Southern Metropolitan,69
Albion,Western Metropolitan,41
Alphington,Northern Metropolitan,34
Altona,Western Metropolitan,74
Altona Meadows,Western Metropolitan,6
Altona North,Western Metropolitan,56


## **RESOLUCIÓN EJERCICIO 1.4: Join**

4. Combinar los datasets de ambas tablas ingestadas utilizando el comando JOIN de SQL  para obtener un resultado similar a lo realizado con Pandas en clase.  


(del)
![](https://i.imgur.com/CUe9Rhy.png)


In [19]:
%%sql joined_tables << 
SELECT mh.*, mhz.*
FROM melbourne_housing mh
JOIN melbourne_housing_by_zipcode mhz ON mh.Postcode = mhz.postcode;

 * sqlite:///melbourne.sqlite3
(sqlite3.OperationalError) no such column: mhz.postcode
[SQL: SELECT mh.*, mhz.*
FROM melbourne_housing mh
JOIN melbourne_housing_by_zipcode mhz ON mh.Postcode = mhz.postcode;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [20]:
joined_tables

NameError: name 'joined_tables' is not defined